### Overview  
Analyzer AI is a Generative AI-powered analytics assistant that can process and interpret raw files (CSV, TXT, JSON, Logs, Excel, etc.) and generate actionable insights in a conversational and visual manner. The system leverages Qwen Model (for natural language understanding and insight generation) and Gradio (for an easy-to-use interface).

---
### Objectives  
- Accept multiple file formats as input.  
- Preprocess and clean the raw data automatically.  
- Analyze datasets using statistical summaries, anomaly detection, and pattern recognition.  
- Generate **human-readable insights** (trends, correlations, recommendations).  
- Provide **visualizations** (charts, graphs, tables) for better interpretability.  
- Enable **interactive querying** through natural language prompts (e.g., “What are the top 5 anomalies?”).  


## Step 1: Project Setup  
- Install required libraries (pandas, numpy, matplotlib/plotly, gradio, openai).  
- Import necessary modules.  
- Configure API keys securely (if using OpenAI API).  
- Ensure GPU is enabled if large models are used (for Hugging Face alternative).  

---

## Step 2: File Upload & Handling  
- Create a Gradio interface for file upload (CSV, Excel, JSON, TXT, log files).  
- Use pandas for reading and storing uploaded data.  
- Implement error handling for unsupported or corrupted files.  

---

## Step 3: Data Preprocessing  
- Clean missing values, handle duplicates, and normalize data.  
- Detect and parse date/time columns automatically.  
- Basic formatting (column renaming, trimming whitespaces, type conversion).  

---

## Step 4: Data Analysis  
- Generate basic statistical summaries (mean, median, mode, std dev).  
- Perform anomaly/outlier detection.  
- Identify correlations and patterns.  
- Summarize key insights (e.g., “Sales peaked in Q3”).  

---

## Step 5: Generative AI Insights  
- Send structured analysis results to OpenAI API.  
- Generate natural language explanations of the data.  
- Provide recommendations and trend summaries (e.g., “Increase marketing budget in Q2 based on rising sales”).  
- Optional: Replace OpenAI with Hugging Face LLMs for free usage.  

---

## Step 6: Data Visualization  
- Generate charts using matplotlib or plotly.  
- Examples: bar chart (category distribution), line chart (time-series trends), scatter plot (correlations), histogram (distribution).  
- Display visualizations within Gradio interface alongside textual insights.  

---

## Step 7: Interactive Querying  
- Allow user to ask natural language questions about the data (e.g., “Show me the top 5 products by sales”).  
- Convert user query → structured data query (via OpenAI or rule-based parsing).  
- Display results as tables, charts, or text.  

---

## Step 8: Gradio Interface Integration  
- Build a user-friendly Gradio dashboard.  
- Sections: File Upload → Data Cleaning → Insights → Visualizations → Query Box.  
- Ensure smooth interaction and responsive results.  

---

## Step 9: Testing & Deployment  
- Test with multiple datasets (CSV, Excel, JSON, logs).  
- Validate correctness of insights and visualizations.  
- Deploy the Gradio app for classroom/demo use.  


In [ ]:
!pip install -q gradio openpyxl huggingface_hub torch transformers accelerate

In [ ]:
import numpy as np
import pandas as pd
import transformers
import matplotlib.pyplot as plt
import os
import re
import gradio as gr
import seaborn as sns
from transformers import pipeline
import torch
from sklearn.ensemble import IsolationForest

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", device_map='auto',torch_dtype='auto')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Step 2: Initialize LLM Helper & Global State
----
This wraps your loaded model and tokenizer into a generator pipeline and defines a helper to handle chat templates.

In [ ]:
generator = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer
)

STATE = {"df": None, 'summary_text': ""}

In [ ]:
def query_qwen(messages, max_tokens=500):
  """Formats chat messages using Qwen's template and generate a response."""
  prompt = tokenizer.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )
  outputs = generator(
      prompt,
      max_new_tokens=max_tokens,
      temperature=0.3,
      do_sample=True,
      pad_token_id=tokenizer.eos_token_id,

  )
  full_text = outputs[0]['generated_text']
  return full_text[len(prompt) :].strip()

# Step 3: Data Ingestion, Cleaning & Statistical Analysis
---
This loads multiple file formats, cleans duplicates/datetimes, runs IsolationForest anomaly detection, and builds statistical context for the LLM.

In [ ]:
def load_and_clean_file(file_obj):
  if file_obj is None:
    return 'No file is Uploaded', None, "", None

  file_path = file_obj.name
  ext = os.path.splitext(file_path)[1].lower()

  try:
    if ext == '.csv':
      df = pd.read_csv(file_path)
    elif ext in ['.xlsx', ".xls"]:
      df = pd.read_excel(file_path)
    elif ext == '.json':
      df = pd.read_json(file_path)
    elif ext in ['.txt', '.log']:
      with open(file_path, "r", encoding = 'utf-8', errors='ignore') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
      df = pd.DataFrame(lines, columns=['raw_log'])
    else:
      return f"Unsupported File Format : {ext}", None, "", None
  except Exception as e:
    return f"Error reading file : {str(e)}", None, "", None

  df.columns = [str(col).strip() for col in df.columns]
  df = df.drop_duplicates().reset_index(drop=True)

  for col in df.columns:
    if df[col].dtype == 'object':
      sample = df[col].dropna().astype(str).head(10)
      if sample.str.contains(r"\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4}").any():
        try:
          df[col] = pd.to_datetime(df[col], errors='coerce')
        except Exception:
          pass

  num_cols = df.select_dtypes(include=[np.number]).columns
  if len(num_cols) > 0:
    df_num = df[num_cols].fillna(df[num_cols].median())
    iso = IsolationForest(contamination=0.05, random_state =42)
    df['is_anomally'] = iso.fit_predict(df_num) == -1
  else:
    df['is_anomally'] = False


  summary_parts = [
      f"Dimensions : {df.shape[0]} rows, {df.shape[1]} columns",
      f"Columns : {list(df.columns)}",
      f"Missing Values :\n{df.isnull().sum().to_dict()}"
  ]
  if len(num_cols) > 0:
    summary_parts.append(
        f"Statistical Summary :\n{df[num_cols].describe().to_string()}"
    )
    summary_parts.append(
        f"Anomalies Detected : {df['is_anomally'].sum()} rows"
    )

  summary_text = "\n\n".join(summary_parts)

  STATE['df'] = df
  STATE['summary_text'] = summary_text

  fig = generate_visualisation(df)

  return (
      f"Successfully processed '{os.path.basename(file_path)}'",
      df.head(10),
      summary_text,
      fig
  )


# Step 4: Data Visualization Engine
---
Generates correlation maps, distributions, or anomaly plots depending on the dataset features.

In [ ]:
def generate_visualisation(df):
  if df is None or df.empty:
    return None

  num_cols = [
      c for c in df.select_dtypes(include=[np.number]).columns if c != 'is_anomally'
  ]
  date_cols = df.select_dtypes(include=['datetime64[ns]']).columns

  fig, axes  = plt.subplots(1, 2, figsize=(11,4))

  if len(num_cols) >=2:
    sns.heatmap(
        df[num_cols].corr(),
        annot=True,
        cmap='mako',
        fmt = '.2f',
        ax = axes[0],
        cbar=False
    )
    axes[0].set_title('Correlation Heatmap')

  elif len(num_cols) ==1:
    sns.histplot(df[num_cols[0]], kde=True, ax=axes[0], color='indigo')
    axes[0].set_title(f"Distribution of {num_cols[0]}")

  else:
    axes[0].text(
        0.5,
        0.5,
        'No Numeric Data',
        ha= 'center',
        va = 'center',
        transform = axes[0].transAxes
    )

  if len(date_cols) > 0 and len(num_cols) > 0:
    time_df = df.sort_values(date_cols[0])
    axes[1].plot(
        time_df[date_cols[0]], time_df[num_cols[0]], color='teal', linewidth=2
    )
    axes[1].set_title(f"{num_cols[0]} Over Time..")
    axes[1].tick_params(axis="x", rotation=45)

  elif "is_anomally" in df.columns:
    counts = df['is_anomally'].value_counts()
    axes[1].bar(
        ['Normal', 'Anomally'],
        [counts.get(False, 0), counts.get(True, 0)],
        color=["#2ecc71", "#e74c3c"]
    )
    axes[1].set_title('Anomally Count')

  else:
    axes[1].text(
        0.5,
        0.5,
        "No Temporal/Anomally Data..",
        ha = 'center',
        va= 'center',
        transform = axes[1].transAxes
    )
  plt.tight_layout()
  return fig

# Step 5: Generative AI Insights & Conversational Chat
---
Connected directly to local Qwen model.

In [ ]:
def generate_ai_insights():
  df = STATE.get('df')
  summary = STATE.get('summary_text')

  if df is None or not summary:
    return 'Please upload and process a file first'

  messages = [
      {
          'role' : 'user',
          'content' : (
              'You are an expert data analyst. Based on dataset summary below, provide:\n'
              "1. Key Trends and Patterns\n"
              "2. Notable Anomalies and Risk\n"
              "3. Top 3 Actionable Recommendations\n\n"
              f"Dataset Summary : \n{summary}"
          )
      }
  ]
  return query_qwen(messages, max_tokens=500)

In [ ]:
def chat_with_dataset(user_message, history):

    df = STATE.get('df')
    summary = STATE.get('summary_text')

    if df is None:
        history.append({
            'role': 'assistant',
            'content': 'Please Upload a Dataset First'
        })
        return history

    system_content = (
        f"You are an Analyzer AI. Answer questions strictly using this data context:\n"
        f"{summary}\n\n"
        f"Sample Records:\n{df.head(5).to_string()}"
    )

    messages = [
        {
            'role': 'system',
            'content': system_content
        }
    ]

    # Add previous chat messages
    messages.extend(history)

    # Add current question
    messages.append({
        'role': 'user',
        'content': user_message
    })

    reply = query_qwen(messages, max_tokens=400)

    # Add response to Gradio history
    history.append({
        'role': 'user',
        'content': user_message
    })

    history.append({
        'role': 'assistant',
        'content': reply
    })

    return history

In [ ]:
#

# Step 6: Assemble and Launch Gradio Dashboard
----

In [ ]:
with gr.Blocks(title='Analyzer AI (Local Qwen)') as demo:
  gr.Markdown("# Analyzer AI Assistant")
  gr.Markdown(
      "**Running Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Local PyTorch Inference)"
  )

  with gr.Tab("1. Ingestion and Analytics"):
    file_input = gr.File(
        label = "Upload Dataset (CSV, XLSX, JSON, LOG, TXT)",
        file_types = [".csv", ".xlsx", ".json", ".log", ".txt"]
    )
    process_btn = gr.Button("Process File", variant="primary")

    status_box = gr.Textbox(label="Status")
    preview_table = gr.Dataframe(label='Cleaned Preview (Top 10 Rows)')
    summary_box = gr.Textbox(label='Data Overview', lines=6)
    plot_box = gr.Plot(label='Visualisations')

  with gr.Tab("2. AI Insights"):
    insights_btn  = gr.Button('Generate AI Insights', variant='primary')
    insights_output = gr.Markdown("Click Above to generate insights..")

  with gr.Tab("3. Natural Language Querying"):
    chatbot = gr.Chatbot(label='Chat with Data', height=350)
    query_input = gr.Textbox(
        placeholder="Ask questions like : 'What are Top Anomalies?",
        label='Your Question'
    )
    clear_btn = gr.Button("Clear Chat")

  # Wire UI Events
  process_btn.click(
      fn=load_and_clean_file,
      inputs = [file_input],
      outputs =[status_box, preview_table, summary_box, plot_box]
  )

  insights_btn.click(fn=generate_ai_insights, outputs=[insights_output])

  query_input.submit(
      fn=chat_with_dataset,
      inputs = [query_input, chatbot],
      outputs=[chatbot]
  ).then(fn=lambda: "",inputs=None, outputs=query_input)

  clear_btn.click(fn=lambda: None, inputs=None, outputs=chatbot, queue=False)


if __name__ == '__main__':
  demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://108f0bda18bdaf6d6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
